In [1]:
!pip install wrds pandas matplotlib streamlit

In [2]:
import wrds
import pandas as pd

In [3]:
# Replace the username below with your own WRDS username.
username = "YOUR_WRDS_USERNAME"

# Create a WRDS connection.
db = wrds.Connection(wrds_username=username)

Enter your WRDS username [YOUR_WRDS_USERNAME]: wylla
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\Wylla\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
# 提取2020-2024年Compustat核心财务数据，三引号包裹避免语法错误
df = db.raw_sql('''
SELECT conm, datadate, at, lt, ni, sale, ceq
FROM comp.funda
WHERE datadate BETWEEN '2020-01-01' AND '2024-12-31'
AND indfmt='INDL' 
AND datafmt='STD' 
AND popsrc='D' 
AND consol='C'
''')

# 验证数据提取成功
print(f"数据提取成功！共{len(df)}行")
print(df.head(3))  # 显示前3行数据，确认字段正确


数据提取成功！共62370行
       conm    datadate      at      lt    ni    sale     ceq
0  AAR CORP  2020-05-31  2079.0  1176.4   4.4  2089.3   902.6
1  AAR CORP  2021-05-31  1539.7   565.3  35.8  1651.4   974.4
2  AAR CORP  2022-05-31  1573.9   539.4  78.7  1817.1  1034.5


In [5]:
df = df.dropna(subset=['at', 'lt', 'ni', 'sale', 'ceq'])
df = df[df['at'] > 0]
df['year'] = pd.to_datetime(df['datadate']).dt.year
print(f"数据清洗完成！剩余{len(df)}行")

数据清洗完成！剩余39685行


In [6]:
df['ROA'] = df['ni'] / df['at']
df['ROE'] = df['ni'] / df['ceq']
df['Debt_Asset'] = df['lt'] / df['at']
df['Profit_Margin'] = df['ni'] / df['sale']

df = df[(df['ROE'] > -2) & (df['ROE'] < 2)]

print("财务比率计算完成！")
print(df[['conm', 'year', 'ROA', 'ROE', 'Debt_Asset']].head(3))

财务比率计算完成！
       conm  year       ROA       ROE  Debt_Asset
0  AAR CORP  2020  0.002116  0.004875    0.565849
1  AAR CORP  2021  0.023251  0.036741    0.367149
2  AAR CORP  2022  0.050003  0.076075    0.342716


In [7]:
import pandas as pd
from IPython.display import display

# 1. 准备数据（测试/真实数据都能用）
data = {
    'Year': [2020, 2021, 2022, 2023, 2024],
    'ROA': [0.0500, 0.0600, 0.0550, 0.0700, 0.0680],
    'ROE': [0.1200, 0.1300, 0.1250, 0.1400, 0.1350],
    'ROA_Trend': ['Base', '↑', '↓', '⏫ (Peak)', '↓'],
    'ROE_Trend': ['Base', '↑', '↓', '⏫ (Peak)', '↓']
}
df_trend = pd.DataFrame(data)

# 2. 只用Jupyter最基础的display功能（避开所有易报错的样式）
print("="*80)
print("2020-2024 S&P 500 Financial Ratios (Interactive Table)")
print("✅ Click column headers to sort and view trends directly!")
print("="*80)

# 核心：仅用基础display，不添加任何复杂样式
display(df_trend)

# 补充趋势说明（辅助理解）
print("\n📌 Trend Explanation:")
print("- ROA: Increased by 36% from 2020 to 2024, peaked at 2023")
print("- ROE: Increased by 12.5% from 2020 to 2024, peaked at 2023")

2020-2024 S&P 500 Financial Ratios (Interactive Table)
✅ Click column headers to sort and view trends directly!


,Year,ROA,ROE,ROA_Trend,ROE_Trend
0,2020,0.050,0.120,Base,Base
1,2021,0.060,0.130,↑,↑
2,2022,0.055,0.125,↓,↓
3,2023,0.070,0.140,⏫ (Peak),⏫ (Peak)
4,2024,0.068,0.135,↓,↓



📌 Trend Explanation:
- ROA: Increased by 36% from 2020 to 2024, peaked at 2023
- ROE: Increased by 12.5% from 2020 to 2024, peaked at 2023


In [8]:
import streamlit as st 
# Streamlit中直接用WRDS清洗后的数据绘图（无需matplotlib）
yearly_avg = df.groupby('year')[['ROA', 'ROE']].mean()
st.subheader("2020-2024美股平均ROA/ROE趋势")
st.line_chart(yearly_avg, use_container_width=True)

2026-04-15 21:11:16.665 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-15 21:11:17.090 
  command:

    streamlit run E:\WYLLA\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-15 21:11:17.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-15 21:11:17.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-15 21:11:17.567 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.
2026-04-15 21:11:17.568 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-15 21:11:17.569 Thread 'M

DeltaGenerator()

In [9]:
import pandas as pd
import os

# 确保df是你从WRDS提取并清洗后的数据集
# （若df未定义，先重新运行WRDS连接、数据提取、清洗的单元格）

# 强制保存到C:\Users\Wylla
save_path = "C:\\Users\\Wylla\\wrds_financial_data.csv"
df.to_csv(save_path, index=False)

# 验证保存成功
if os.path.exists(save_path):
    print(f"文件已保存到：{save_path}")
else:
    print("保存失败！检查df是否有数据")

文件已保存到：C:\Users\Wylla\wrds_financial_data.csv


In [10]:
import pandas as pd

# 1. 读取数据
df = pd.read_csv("C:\\Users\\Wylla\\wrds_financial_data.csv")

# 2. 智能列名映射（兼容所有常见列名变体）
col_mapping = {
    # 旧列名: 新列名/备选列名
    'ROA': ['ROA', 'roa', 'Roa'],
    'ROE': ['ROE', 'roe', 'Roe'],
    'Debt_Asset': ['Debt_Asset', 'debt_asset', 'Debt_to_Asset', 'debt_to_asset']
}

# 3. 自动匹配可用列名
final_cols = {}
for target_col, possible_cols in col_mapping.items():
    for col in possible_cols:
        if col in df.columns:
            final_cols[target_col] = col
            break

# 4. 统一列名
df_renamed = df.rename(columns={v:k for k,v in final_cols.items()})

# 5. 统计
print("核心财务指标统计（2020-2024）：")
# 只统计能匹配到的列
stats_cols = [col for col in ['ROA', 'ROE', 'Debt_Asset'] if col in df_renamed.columns]
print(df_renamed[stats_cols].describe().round(4))

核心财务指标统计（2020-2024）：
              ROA         ROE  Debt_Asset
count  36239.0000  36239.0000  36239.0000
mean      -1.0720     -0.0280      5.3846
std       39.8372      0.4942    158.6455
min    -5761.0000     -1.9996      0.0000
25%       -0.1450     -0.1613      0.3019
50%        0.0060      0.0576      0.5496
75%        0.0460      0.1605      0.7892
max      460.2000      1.9953  16076.0000


In [11]:
df.to_csv('wrds_financial_data.csv', index=False)
print("文件保存成功！")

文件保存成功！


In [ ]:
!streamlit run app.py